In [7]:
import os
import numpy as np
import matplotlib.pyplot as plt
import cv2
from scipy import io
from cv2.ml import KNearest_create
from google.colab import files

In [ ]:
os.makedirs('dataset', exist_ok=True)

uploaded = files.upload()

for filename in uploaded.keys():
  dest = os.path.join('dataset', filename)
  os.rename(filename, dest)
  print(f"File '{filename}' Moved To '{dest}'.")

print("\nContents Of Dataset Folder:")
print(os.listdir('dataset'))

Saving Data_hoda_full.mat to Data_hoda_full.mat
File 'Data_hoda_full.mat' Moved To 'dataset/Data_hoda_full.mat'.

Contents Of Dataset Folder:
['Data_hoda_full.mat']


In [4]:
def load_hoda(training_sample_size = 1000, test_sample_size = 200, size = 5):
  dataset = io.loadmat('./dataset/Data_hoda_full.mat')

  x_train_original = np.squeeze(dataset['Data'][:training_sample_size])
  y_train = np.squeeze(dataset['labels'][:training_sample_size])
  x_test_original = np.squeeze(dataset['Data'][training_sample_size:training_sample_size + test_sample_size])
  y_test = np.squeeze(dataset['labels'][training_sample_size:training_sample_size + test_sample_size])

  x_train_5by5 = [cv2.resize(img, dsize=(size, size)) for img in x_train_original]
  x_test_5by5 = [cv2.resize(img, dsize=(size, size)) for img in x_test_original]

  x_train = np.reshape(x_train_5by5, [-1, size ** 2])
  x_test = np.reshape(x_test_5by5, [-1, size ** 2])

  return x_train, y_train, x_test, y_test

In [5]:
x_train, y_train, x_test, y_test = load_hoda()

x_train.shape, y_train.shape, x_test.shape, y_test.shape

((1000, 25), (1000,), (200, 25), (200,))

In [6]:
x_train = x_train.astype('float32')
y_train = y_train.astype('int64').reshape(-1 , 1)

print(x_train.shape, y_train.shape)

(1000, 25) (1000, 1)


In [8]:
knn = KNearest_create()
knn.train(x_train, cv2.ml.ROW_SAMPLE, y_train)

True

In [9]:
rest, result, neighbors, dist = knn.findNearest(x_test.astype('float32'), k = 3)

print(result.ravel())

[7. 2. 3. 1. 5. 5. 4. 7. 3. 2. 0. 8. 8. 0. 3. 9. 3. 6. 7. 4. 0. 3. 6. 3.
 9. 2. 7. 5. 2. 9. 7. 5. 5. 8. 9. 6. 5. 1. 4. 8. 8. 4. 2. 7. 1. 2. 7. 9.
 0. 3. 7. 4. 7. 5. 2. 9. 8. 2. 9. 8. 8. 6. 6. 6. 7. 6. 2. 4. 1. 4. 4. 5.
 9. 1. 8. 2. 0. 5. 6. 2. 4. 3. 2. 7. 7. 7. 7. 1. 8. 1. 7. 8. 7. 7. 8. 9.
 3. 2. 3. 1. 0. 2. 9. 7. 3. 5. 5. 0. 0. 2. 6. 7. 9. 3. 9. 9. 8. 7. 9. 2.
 5. 2. 5. 5. 9. 6. 9. 2. 0. 3. 7. 6. 5. 2. 9. 0. 4. 1. 8. 2. 2. 3. 0. 2.
 9. 3. 8. 6. 7. 0. 9. 9. 0. 7. 6. 1. 4. 7. 9. 3. 7. 0. 7. 1. 9. 4. 7. 3.
 4. 1. 5. 6. 7. 9. 1. 3. 5. 4. 5. 7. 4. 1. 3. 3. 1. 2. 5. 3. 8. 9. 6. 7.
 7. 2. 3. 0. 1. 4. 9. 5.]


In [10]:
matches = np.equal(result.ravel(), y_test)

np.mean(matches)

np.float64(0.96)